In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

Libraries imported successfully!


In [13]:
import os
import glob
import pandas as pd


target_name = "train.csv"
file_path = None

if os.path.exists(target_name):
    file_path = target_name
else:
    
    matches = glob.glob(f"E:/**/{target_name}", recursive=True)
    if matches:
        file_path = matches[0]
    else:
        
        user_home = os.path.expanduser("~")
        dl_matches = glob.glob(f"{user_home}/Downloads/**/{target_name}", recursive=True)
        if dl_matches:
            file_path = dl_matches[0]


if file_path:
    print(f"File found at: {file_path}")
    df = pd.read_csv(file_path)
    df_original = df.copy()

    print("\n=== DATA QUALITY REPORT (BEFORE CLEANING) ===")
    print("1. Shape of Dataset (Rows, Columns):", df.shape)
    print("\n2. Duplicate Rows Count:", df.duplicated().sum())

    print("\n3. Missing Values per Column:")
    null_summary = pd.DataFrame({
        'Missing Values': df.isnull().sum(),
        'Percentage (%)': (df.isnull().sum() / len(df)) * 100
    })
    display(null_summary)

    print("\n4. Data Types and Non-Null Counts:")
    df.info()

    display(df.head())
else:
    print("Error: 'train.csv' not found in the current directory or Downloads folder.")

Error: 'train.csv' not found in the current directory or Downloads folder.


In [14]:
import pandas as pd
import numpy as np


url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
print("Downloading and loading Titanic dataset...")
df = pd.read_csv(url)


df_original = df.copy()

print("Data loaded successfully! Shape:", df.shape)


duplicates_before = df.duplicated().sum()
df = df.drop_duplicates()

print("=== DUPLICATE REMOVAL STATUS ===")
print(f"Total duplicate rows removed: {duplicates_before}")
print(f"Remaining total rows: {len(df)}")
df.head()

Data loaded successfully! Shape: (891, 12)
=== DUPLICATE REMOVAL STATUS ===
Total duplicate rows removed: 0
Remaining total rows: 891


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [15]:

df['Sex'] = df['Sex'].astype(str).str.strip().str.capitalize()


df['PassengerId'] = df['PassengerId'].astype(str)


df['Pclass'] = df['Pclass'].astype('category')

print("Sex column unique values after standardisation:", df['Sex'].unique())
print("Data types updated:")
print(df[['PassengerId', 'Pclass', 'Sex']].dtypes)

Sex column unique values after standardisation: <ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str
Data types updated:
PassengerId         str
Pclass         category
Sex                 str
dtype: object


In [16]:

df.drop(columns=['Cabin'], inplace=True)
print("Dropped 'Cabin' column due to excessive missing values (>75%).")


age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)
print(f"Filled missing 'Age' using Median value: {age_median}")


embarked_mode = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(embarked_mode)
print(f"Filled missing 'Embarked' using Mode value: '{embarked_mode}'")

print("\nRemaining Missing Values:\n", df.isnull().sum())

Dropped 'Cabin' column due to excessive missing values (>75%).
Filled missing 'Age' using Median value: 28.0
Filled missing 'Embarked' using Mode value: 'S'

Remaining Missing Values:
 PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [17]:

numeric_cols = ['Fare', 'Age']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Detected {len(outliers)} outliers in '{col}' [Lower: {lower_bound:.2f}, Upper: {upper_bound:.2f}]")
    
   
    df[col] = np.clip(df[col], lower_bound, upper_bound)

print("\nOutliers successfully capped within realistic business bounds!")

Detected 116 outliers in 'Fare' [Lower: -26.72, Upper: 65.63]
Detected 66 outliers in 'Age' [Lower: 2.50, Upper: 54.50]

Outliers successfully capped within realistic business bounds!


In [18]:
summary_data = {
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Missing Values Count',
        'Duplicate Rows Count',
        'Max Age (Outlier check)',
        'Max Fare (Outlier check)'
    ],
    'Before Cleaning': [
        df_original.shape[0],
        df_original.shape[1],
        df_original.isnull().sum().sum(),
        df_original.duplicated().sum(),
        df_original['Age'].max(),
        df_original['Fare'].max()
    ],
    'After Cleaning': [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        round(df['Age'].max(), 2),
        round(df['Fare'].max(), 2)
    ]
}

comparison_table = pd.DataFrame(summary_data)
print("=== BEFORE VS. AFTER CLEANING SUMMARY ===")
display(comparison_table)

=== BEFORE VS. AFTER CLEANING SUMMARY ===


,Metric,Before Cleaning,After Cleaning
0,Total Rows,891.0000,891.00
1,Total Columns,12.0000,11.00
2,Missing Values Count,866.0000,0.00
3,Duplicate Rows Count,0.0000,0.00
4,Max Age (Outlier check),80.0000,54.50
5,Max Fare (Outlier check),512.3292,65.63


In [19]:

output_file = "cleaned_titanic_dataset.csv"
df.to_csv(output_file, index=False)

print(f"Cleaned dataset saved successfully as '{output_file}'!")

Cleaned dataset saved successfully as 'cleaned_titanic_dataset.csv'!


#  Task 3: Data Cleaning Decisions & Justifications

### 1. Handling Missing Data:
* **Cabin Column:** Dropped entirely because over 75% of values were missing. Imputing such a high percentage would introduce heavy bias into any downstream analysis.
* **Age Column:** Imputed missing values using the **Median** ($28$ years) instead of Mean. Age has positive skewness and outliers, making the median a much more robust central tendency measure.
* **Embarked Column:** Imputed missing entries using the **Mode** (`'S'`), as it represents categorical data and Southampton was the most frequent embarkation port.

### 2. Standardization & Data Types:
* **Text Normalization:** Stripped leading/trailing whitespaces and capitalized values in categorical fields like `Sex` to maintain consistency.
* **Type Casting:** Converted `PassengerId` to string/object because IDs are unique identifiers and cannot undergo mathematical calculations. Converted `Pclass` to categorical type.

### 3. Outlier Handling (IQR Method):
* Identified outliers in `Fare` and `Age` using the Interquartile Range ($1.5 \times IQR$).
* Rather than dropping rows (which would cause information loss), outliers were **capped (clipped)** at the upper and lower bounds to maintain statistical integrity.

### 4. Output:
* Successfully exported the transformed and cleaned dataset to `cleaned_titanic_dataset.csv`.

In [20]:

cleaned_df = pd.read_csv("cleaned_titanic_dataset.csv")
print("Cleaned Dataset Shape:", cleaned_df.shape)
print("Remaining Nulls:", cleaned_df.isnull().sum().sum())
print("Duplicates:", cleaned_df.duplicated().sum())
cleaned_df.head()

Cleaned Dataset Shape: (891, 11)
Remaining Nulls: 0
Duplicates: 0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",Male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Female,38.0,1,0,PC 17599,65.6344,C
2,3,1,3,"Heikkinen, Miss. Laina",Female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",Male,35.0,0,0,373450,8.0500,S
